In [1]:
library(dplyr)
library(ggplot2)
library(data.table)
library(readxl)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [2]:
communities_ids = fread("PRJNA786578_annotate.tsv")[1:5, c("run", "sample_name")] |>
    rename(community = run, StudyID = sample_name)
tools = c("sylph", "singlem-regime3", "metax")
d1 = data.table(expand.grid(tool = tools, community = communities_ids$community))
readit = function(tool, community){
    to_read = paste('output_',tool,sep='','/', tool, '/',community,'.profile')
    # print(to_read)
    return(fread(to_read, colClasses = list(double = "coverage", character = c("taxonomy", "sample")))[,c("sample", "taxonomy", "coverage")])
}
d2 = communities_ids[d1[, readit(tool, community), by=list(tool,community)], on = "community"]

ERROR: Error in fread(to_read, colClasses = list(double = "coverage", character = c("taxonomy", : File 'output_metax/metax/SRR17156932.profile' does not exist or is non-readable. getwd()=='/mnt/hpccs01/home/lamber22/git/singlem-benchmarking/19_clinical_stool'


In [ ]:
d[1:5,]

community,StudyID,tool,sample,taxonomy,coverage
<chr>,<chr>,<fct>,<chr>,<chr>,<dbl>
SRR17156932,19_0036,sylph,SRR17156932,d__Bacteria;p__Firmicutes_C;c__Negativicutes;o__Veillonellales;f__Dialisteraceae;g__Dialister;s__Dialister invisus,367.474
SRR17156932,19_0036,sylph,SRR17156932,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Lachnospirales;f__Lachnospiraceae;g__Agathobacter;s__Agathobacter rectalis,155.122
SRR17156932,19_0036,sylph,SRR17156932,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__Oscillospirales;f__Ruminococcaceae;g__Faecalibacterium;s__Faecalibacterium prausnitzii_G,114.459
SRR17156932,19_0036,sylph,SRR17156932,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Bacteroidaceae;g__Bacteroides;s__Bacteroides uniformis,48.193
SRR17156932,19_0036,sylph,SRR17156932,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Bacteroidales;f__Bacteroidaceae;g__Phocaeicola;s__Phocaeicola vulgatus,43.675


In [ ]:
res = d2[
    coverage > 0,
    list(
        Campy = any(grepl("g__Campylobacter", taxonomy)),
        Salm = any(grepl("g__Salmonella", taxonomy))
    ),
    by = c("community", "StudyID", "tool", "sample")
] |>
    melt(value.name = "tool_detected", variable.name = "target", measure.vars = c("Campy", "Salm"))

In [ ]:
res[1:5,]

community,StudyID,tool,sample,target,tool_detected
<chr>,<chr>,<fct>,<chr>,<fct>,<lgl>
SRR17156932,19_0036,sylph,SRR17156932,Campy,TRUE
SRR17156932,19_0036,singlem-regime3,SRR17156932.1,Campy,TRUE
SRR17156932,19_0036,metax,SRR17156932,Campy,TRUE
SRR17156933,19_0051,sylph,SRR17156933,Campy,TRUE
SRR17156933,19_0051,singlem-regime3,SRR17156933.1,Campy,TRUE


In [ ]:
meta <- read_xlsx("metadata/Dataset 1_Results Summary.xlsx", skip = 1) |>
    filter(!is.na(StudyID)) |>
    mutate(
        pick("StudyID"),
        Campy = CultureSummary %in% c("Campy", "Campy and STEC"),
        Salm = CultureSummary %in% "Salm",
        healthy = CultureSummary %in% "healthy",
        neg = CultureSummary %in% "neg",
        .keep = "none"
    ) |>
    data.table() |>
    melt(
        measure.vars = c("Campy", "Salm", "neg"),
        value.name = "culture_detected",
        variable.name = "target"
    )

New names:
• `` -> `...39`
• `` -> `...40`
• `` -> `...41`
• `` -> `...42`
• `` -> `...43`
• `` -> `...44`
• `` -> `...45`
• `` -> `...46`
• `` -> `...47`


In [ ]:
meta[1:5,]

StudyID,healthy,target,culture_detected
<chr>,<lgl>,<fct>,<lgl>
14_0001,FALSE,Campy,FALSE
14_0002,FALSE,Campy,TRUE
14_0003,FALSE,Campy,TRUE
14_0004,FALSE,Campy,FALSE
14_0005,FALSE,Campy,FALSE


In [ ]:
eval = meta[res, on = c("StudyID", "target")][
    ,
    result := case_when(
        culture_detected & tool_detected ~ "TP",
        culture_detected & !tool_detected ~ "FN",
        !culture_detected & tool_detected ~ "FP",
        !culture_detected & !tool_detected ~ "TN"
    )
]

In [ ]:
eval2 = eval[
    ,
    as.list(table(factor(result, c("TP", "TN", "FP", "FN")))),
    by = tool
][
    ,
    `:=`(
        Sens. = TP / (TP + FN),
        Spec. = TP / (TP + FP)
    )
][, F1 := 2 * (Sens. * Spec.) / (Sens. + Spec.)]

In [ ]:
eval2

tool,TP,TN,FP,FN,Sens.,Spec.,F1
<fct>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>
sylph,1,6,3,0,1,0.2500000,0.4000000
singlem-regime3,1,7,2,0,1,0.3333333,0.5000000
metax,1,8,1,0,1,0.5000000,0.6666667
